In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from joblib import Parallel, delayed
sys.path.insert(0, "..")
sys.path.insert(0, ".")
from support.paths import resolve


In [ ]:
"""
Fast high-accuracy NEM price forecaster.

Method
------
Spike-aware direct multi-horizon LightGBM ensemble with six components
per forecast horizon:

  1. BASE L1 model   – regression_l1 on arcsinh(clip(y, p97)). Clipping the
                       target at the 97th percentile removes extreme-spike
                       noise from the base model's loss, giving it a clean
                       training signal for the majority of intervals.

  2. BASE L2 model   – regression (MSE) on arcsinh(y), uncapped target.
                       Blending L1+L2 (Lago et al. 2021) reduces both MAE
                       and RMSE simultaneously. Per-horizon blend weight α
                       tuned on the validation set.

  3. SPIKE classifier – binary LightGBM estimating P(price > SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  4. SPIKE regressor  – regression_l1 on arcsinh(y) (full range, no clipping).
                        Spike rows receive 10x sample weight.

  5. SPIKE quantile   – P90 quantile regression on arcsinh(y) for a
                        conservative spike ceiling estimate.

  6. DIP classifier + regressor – mirrors spike components for negative/
                        near-zero prices (y < DIP_THRESHOLD).

Final prediction pipeline per horizon:
  a. Blend L1 + L2 base predictions (α tuned on validation).
  b. Apply spike policy (soft blend / gated uplift / hard gate — kind and
     parameters tuned on validation via _spike_policy_score).
  c. Apply dip policy (same three kinds, tuned independently).
  d. Blend model prediction with lag-naive baseline (α tuned on validation).
  e. Isotonic regression calibration fitted on validation set.

Additionally:
  - Per-horizon target-time features (8 feats: hour/dow sin-cos + regime flags)
  - Cross-features (4 feats: doy cycle, SA-NSW spread, region spike count)
  - Exponential recency weights (newest row ~4.5x oldest)

Config reads TARGET, FEATURE_DATASET, HORIZON_HOURS, OUTPUT_RESOLUTION from
environment variables set by 0_Config/0_variables.ipynb.

Based on Lago et al. (2021) review of state-of-the-art EPF methods and
Ziel & Weron (2018) on spike-aware electricity price forecasting.
"""


In [ ]:
# Model configuration
# PRICE_TRANSFORM_SCALE = 100.0
# SPIKE_THRESHOLD = 150.0
# _DIP_THRESHOLD = 0.0
# BASE_CLIP_PERCENTILE = 97.0    # raised 95→97: include more of the spike distribution
# _SPIKE_UPWEIGHT = 10.0  # spike row weight multiplier (raised 5→10)
# _DIP_UPWEIGHT = 7.0   # raised 4→7 # Low-price specialist: learn downside tails (negative/near-zero prices).
# _MIN_SPIKE_TRAIN = 20  # minimum spike rows needed to train spike chain per horizon

In [ ]:
# EARLY_STOPPING_ROUNDS = 75       # reduced 150→75 for faster convergence
# SPIKE_ES_ROUNDS = 50       # reduced 100→50

In [ ]:


# Model hyperparameters 
# from parameters import (
#     LGBM_BASE_PARAMS,
#     LGBM_BASE_L2_PARAMS,
#     LGBM_CLF_PARAMS,
#     LGBM_SPIKE_PARAMS,
#     LGBM_SPIKE_Q_PARAMS,
#     LGBM_DIP_CLF_PARAMS,
#     LGBM_DIP_PARAMS,
    # _SPIKE_THR_GRID,
    # _SPIKE_POW_GRID,
    # _SPIKE_WMAX_GRID,
    # _SPIKE_GATE_W_GRID,
    # _DIP_THR_GRID,
    # _DIP_POW_GRID,
    # _DIP_WMAX_GRID,
    # _DIP_GATE_W_GRID,
# )
